# Document Dewarping — training on Kaggle

Predicts a dense backward flow field that rectifies a photographed page, with a
**layout-consistency loss** built on PubLayNet's region annotations: a correctly
dewarped text block must come out as an axis-aligned rectangle.

## Before you run anything

**Settings → Accelerator: GPU T4 x2** and **Internet: off.**

**Add Data →** attach all three:

| dataset | size | what it is |
|---|---|---|
| PubLayNet Dewarping Code | ~0.3 MB | the `.py` modules, flattened |
| PubLayNet Dewarping Layout Store | 57 MB | region annotations for 47,724 pages |
| PubLayNet Dewarping Dataset | 25 GB | 145,313 warped pages + flow + masks |

---

## Which files survive, and which do not

This decides how you run the notebook, so it comes before anything else.

| path | writable | survives the session |
|---|---|---|
| `/kaggle/input/...` | no | always there, never changes |
| `/kaggle/working/` | **yes** | **only if you save a version** |
| `/kaggle/temp/`, `/tmp`, anywhere else | yes | **never** — gone without warning |

`/kaggle/working` is the only place worth writing to, and writing there is not
the same as keeping it. A session that is closed, that idles out, or that hits
Kaggle's 12-hour kill **saves nothing at all**. Checkpoints are turned into
something permanent only by *Save Version*, which has two modes and they behave
very differently:

- **Save & Run All (Commit)** — starts a **fresh batch session** and re-runs the
  notebook from the top. What that batch run writes to `/kaggle/working` becomes
  the saved output. It does **not** photograph your current interactive session.
- **Quick Save** — saves the notebook and, with output included, the files
  currently in `/kaggle/working` from the interactive session.

**The consequence that costs a session:** training interactively for 11 hours and
*then* pressing Save & Run All throws that work away and starts an identical
11-hour run in the background. Do one or the other, not both.

**Recommended:** run sections 1–4 interactively (about three minutes) to check
everything is mounted and healthy, set `MODE = 'session1'`, then **Save & Run
All (Commit)** and let the batch session do the training. That is what the
`MODE` switch in section 5 exists for — *Run All* must perform exactly one training
run, never all of them.

Once the version finishes, its output is attachable to the next notebook as an
input, which is how session N+1 finds session N's checkpoints.

A note on **Settings → Persistence**: "Files only" / "Variables and Files" keeps
`/kaggle/working` across *interactive* restarts. Convenient, but it is not what
you attach to the next session, and it is not worth betting an 11-hour run on.
Saved versions are the mechanism that matters.

Output is capped at roughly 20 GB. This run stays near **300 MB** — checkpoints
are ~78 MB and pruned to the last two plus `best.pt` plus `last.pt` — so size is
not a risk here.

---

## One run spans several sessions

Kaggle kills a session at 12 hours. 30 epochs took two sessions per arm on a T4 (about 8.6 steps/s),
so the run *must* survive being stopped and restarted. It does: `--max_hours 11.0` exits
cleanly with a checkpoint inside the limit, and session N+1 picks it up with
`--resume_dirs auto`. Resume restores weights, optimizer moments, AMP scaler,
LR-schedule position, epoch, global step, position *within* the epoch, and all
RNG states — verified bit-identical against an uninterrupted run in a CPU test.

## 1 — Environment

In [ ]:
import subprocess, sys, os, shutil
try:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                          '--format=csv'], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print('no nvidia-smi — this session has NO GPU')

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.device_count(), 'device(s)')
total, used, free = shutil.disk_usage('/kaggle/working')
print(f'/kaggle/working free: {free/2**30:.1f} GiB of {total/2**30:.1f} GiB')

# Forgetting the accelerator is easy and expensive: training would silently run
# on CPU at roughly 1/50th the speed and the session would end having covered a
# few hundred steps.
assert torch.cuda.is_available(), \
    'No GPU. Settings -> Accelerator -> GPU T4 x2, then rerun.'

## 2 — Find where Kaggle mounted the datasets

**Do not hard-code a mount path.** Kaggle mounts an attached dataset at
`/kaggle/input/<slug>` in some notebooks and at
`/kaggle/input/datasets/<owner>/<slug>/versions/N` in others, and nothing outside
the notebook tells you which you will get. A literal path fails with

```
cp: cannot stat '/kaggle/input/publaynet-dewarp-code/*.py': No such file or directory
```

while the Input panel shows the dataset attached and perfectly healthy — an error
that points at the dataset when the broken thing is the path. Locate each store
by a file that is known to live inside it instead.

In [ ]:
import os
from pathlib import Path

def find_dir(marker, depth=7):
    '''Every directory under /kaggle/input that directly contains `marker`.'''
    base = Path('/kaggle/input')
    hits = []
    for root, dirs, files in os.walk(base):
        if len(Path(root).relative_to(base).parts) > depth:
            dirs[:] = []
            continue
        if marker in files:
            hits.append(root)
    return sorted(hits)

CODE   = (find_dir('train.py')    or [''])[0]   # the code dataset
LAYOUT = (find_dir('stems.jsonl') or [''])[0]   # the layout store
DATA   = (find_dir('flows.f16')   or [''])[0]   # the big dataset

for name, p in [('CODE', CODE), ('LAYOUT', LAYOUT), ('DATA', DATA)]:
    print(f'{name:7s} {p or "** NOT FOUND — is it attached? **"}')

assert CODE and LAYOUT and DATA, 'Attach the missing dataset(s), then rerun this cell.'

## 3 — Check the 25 GB upload arrived intact

An 11-hour run that dies at hour 6 on a truncated file is the most expensive
failure available here, and it costs two seconds to rule out. The flow and mask
blobs are fixed-stride, so their exact byte length is predictable from
`meta.json` — if the arithmetic works, the upload is complete.

In [ ]:
import json
meta = json.load(open(Path(DATA) / 'meta.json'))
n, fw, fh = meta['n_samples'], meta['flow_w'], meta['flow_h']
print(f"{n:,} samples | page {meta['page_w']}x{meta['page_h']} | flow {fw}x{fh}")

expect = {'flows.f16': n * fh * fw * 2 * 2,      # 2 channels, float16
          'masks.u8':  n * fh * fw}
ok = True
for name, want in expect.items():
    got = (Path(DATA) / name).stat().st_size
    good = got == want
    ok &= good
    print(f"  {name:12s} {got:>15,} bytes  expected {want:>15,}  "
          f"{'OK' if good else '** TRUNCATED **'}")

shards = sorted(Path(DATA).glob('pages-*.bin'))
cleans = sorted(Path(DATA).glob('clean-*.bin'))
print(f"  {len(shards)} page shards, {len(cleans)} clean shards, "
      f"{sum(p.stat().st_size for p in shards + cleans)/2**30:.2f} GiB of JPEG")

for f in ('manifest.jsonl', 'clean_index.jsonl'):
    print(f"  {f:20s} {sum(1 for _ in open(Path(DATA)/f)):,} rows")

assert ok, 'A blob is the wrong size — re-upload the dataset.'
print('\nDATASET OK')

## 4 — Stage the code, and confirm it is the current version

A Kaggle notebook **pins the dataset version it was attached at.** Re-upload the
code and this notebook keeps silently using the old copy until you re-add it. The
`find_under` check below is the tell: it only exists in the version that resolves
mount paths itself.

In [ ]:
import shutil
src = (Path(CODE) / 'train.py').read_text()
if 'def find_under' not in src:
    raise SystemExit(
        'This notebook is pinned to an OLD version of the code dataset.\n'
        'Input panel -> remove the Code dataset -> Add Data -> re-add it, '
        'then rerun cells 2 and 4.')

WORK = Path('/kaggle/working/code')
WORK.mkdir(parents=True, exist_ok=True)
for p in sorted(Path(CODE).glob('*.py')):
    shutil.copy2(p, WORK / p.name)
print(f'code version OK — {len(list(WORK.glob("*.py")))} modules staged in {WORK}')
print('  ' + ', '.join(sorted(p.name for p in WORK.glob('*.py'))))

## 5 — Pick what this session does

**One switch, because *Run All* must perform exactly one training run.** A
notebook with four unguarded training cells would, on commit, try to run all
four in a single 12-hour session and finish none of them.

| `MODE` | what it does | when |
|---|---|---|
| `'smoke'` | 50 steps, ~2 min | **always run this first**, interactively |
| `'session1'` | the layout arm, from scratch | the first real session |
| `'resume'` | the layout arm, continuing | every later session of that arm |
| `'ablation'` | the control arm (`--w_layout 0`), from scratch | first session of the ablation |
| `'ablation_resume'` | the control arm, continuing | every later session of the ablation |

The two `_resume` modes need the previous session's saved output attached as an
input; they find the checkpoint themselves.

**Attach only one previous run at a time.** `'resume'` and `'ablation_resume'`
both search `/kaggle/input` for a `last.pt`, so with both arms attached at once
they could pick up the wrong one and quietly cross-contaminate the experiment.

In [ ]:
MODE = 'smoke'   # 'smoke' | 'session1' | 'resume' | 'ablation' | 'ablation_resume'

EPOCHS      = 30        # both arms MUST use the same number
TOTAL_STEPS = 540_000   # the LR schedule horizon. IDENTICAL in every session of
                        # both arms, or the cosine restarts and the runs stop
                        # being comparable. To train longer, raise EPOCHS and
                        # TOTAL_STEPS together (17,733 steps per epoch) and do it
                        # to BOTH arms.
BATCH   = 8             # drop to 6 if CUDA OOMs
HOURS   = 11.0          # clean exit inside Kaggle's 12-hour kill
WORKERS = 4

## 6 — Run

Streams the child's output live, in interactive sessions and in committed batch
runs alike (`python -u`, because a piped child otherwise buffers and you would
watch a blank cell for an hour).

In [ ]:
import subprocess, sys, time

PRESETS = {
    'smoke':    dict(out='smoke',        extra=['--steps', '50', '--max_hours', '0.2']),
    'session1': dict(out='run_layout',   extra=['--epochs', str(EPOCHS),
                                                '--max_hours', str(HOURS)]),
    'resume':   dict(out='run_layout',   extra=['--epochs', str(EPOCHS),
                                                '--max_hours', str(HOURS),
                                                '--resume_dirs', 'auto']),
    'ablation': dict(out='run_nolayout', extra=['--epochs', str(EPOCHS),
                                                '--max_hours', str(HOURS),
                                                '--w_layout', '0']),
    'ablation_resume': dict(out='run_nolayout', extra=['--epochs', str(EPOCHS),
                                                '--max_hours', str(HOURS),
                                                '--w_layout', '0',
                                                '--resume_dirs', 'auto']),
}
assert MODE in PRESETS, f'MODE must be one of {list(PRESETS)}'
cfg = PRESETS[MODE]
OUT = f"/kaggle/working/{cfg['out']}"

cmd = [sys.executable, '-u', 'train.py',
       '--root', 'auto', '--layout_root', 'auto',
       '--out', OUT,
       '--batch', str(BATCH), '--total_steps', str(TOTAL_STEPS),
       '--workers', str(WORKERS)] + cfg['extra']

print(f'MODE = {MODE}\n$ ' + ' '.join(cmd) + '\n', flush=True)
t0 = time.time()
proc = subprocess.Popen(cmd, cwd='/kaggle/working/code', text=True, bufsize=1,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\n[exit {proc.returncode} after {(time.time()-t0)/60:.1f} min]', flush=True)
if proc.returncode != 0:
    raise SystemExit(f'training exited {proc.returncode} — read the traceback above')

### What to read in a smoke test, in this order

1. `train 141,869 / val 2,737` — both stores loaded, the 707-sample exclusion
   applied. Near zero means the wrong stores are mounted.
2. `[val @ start] epe ~37 px` — an untrained network scores about what the
   identity warp scores. Much lower means something is leaking; much higher means
   the labels are wrong.
3. `epe_px` falling across the 50 steps.
4. No CUDA OOM. If it OOMs: `BATCH = 6`, then add `--base 24`.

**Write down the steps/s.** We measured 8.6 steps/s on a T4, which made 30 epochs two sessions per
arm; a much lower number means something is throttling the run.

In `'resume'` mode, check the first three lines instead:

```
[paths] resume candidate /kaggle/input/.../run_layout
[resume] last.pt: epoch 3, step 54,231, 12,345 samples into the epoch, best epe 4.12
[val @ start] epe 4.12px
```

If it says **`STARTING FROM SCRATCH`**, stop immediately — the previous output is
not attached, and letting it run burns the whole session on a restart.

### Two failures with their own messages

**`EMPTY SPLIT`** — the layout store and the image dataset do not belong together
(wrong dataset, or a stale version). It prints the stem overlap and examples from
each side.

**`pinned to an OLD version of the code dataset`** — see cell 4.

## 7 — What this session produced, and whether it is safe

In [ ]:
import json
from pathlib import Path
run = Path(OUT)
tot = 0
for p in sorted(run.iterdir()):
    sz = p.stat().st_size
    tot += sz
    print(f'  {p.name:24s} {sz/2**20:>8.1f} MiB')
print(f'  {"TOTAL":24s} {tot/2**20:>8.1f} MiB   (output cap is ~20 GB)')

log = run / 'log.jsonl'
if log.exists():
    rows = [json.loads(l) for l in open(log)]
    val = [r for r in rows if r.get('split') in ('val', 'final')]
    print(f'\n{len(rows)} log rows, {len(val)} validations')
    for r in val[-8:]:
        print(f"  step {r['step']:>8,}  epe {r.get('epe_px', float('nan')):7.3f} px"
              f"  layout {r.get('layout', float('nan')):.4f}")

print('\nThese files are in /kaggle/working and are NOT saved yet.')
print('Save Version -> they become this notebook version\'s output.')
print('Attach that output as an input next session and run with MODE = "resume".')

## 8 — Sessions 2, 3, 4 …

1. Confirm the previous session's **version finished** and has an output. If it
   was killed at the 12-hour mark, there is nothing saved and that session is
   lost — which is exactly what `--max_hours 11.0` exists to prevent.
2. *Add Data* → *Your Work* (or *Your Datasets*) → that saved output.
3. Section 5: `MODE = 'resume'`. **Nothing else changes** — same `TOTAL_STEPS`, same
   `BATCH`, same output directory.
4. Run sections 1–4 to verify the mounts, then **Save & Run All (Commit)**.

`'resume'` searches `/kaggle/input` for a `last.pt` and prints every candidate,
so the mount-path problem cannot bite here either. Attach only one previous run
at a time and there is nothing to disambiguate.

## 9 — The ablation

The experiment the paper rests on, so it must differ in **exactly one thing**:
the layout loss weight. Same seed, same schedule, same data, its own output
directory, its own continuation chain: `MODE = 'ablation'` for its first
session, `'ablation_resume'` for each one after.

Compare `epe_px` from each run's `log.jsonl`. The ablation never computes the
layout term, so compare layout distortion offline for both arms with
`training/inference.py --full_val` (see docs/REPRODUCE.md).